## 1. Imports and Environment Setup

In [1]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import numpy as np

import emoji
import regex
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

SEED = 37
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import warnings
warnings.filterwarnings(action='ignore', message='FutureWarning')

In [2]:
# # Download necessary NLTK corpora
# nltk.download('wordnet', quiet=True)
# nltk.download('omw-1.4', quiet=True)
# print("NLTK resources checked and downloaded successfully!")

## 2. Data Loading and Cleaning

In [3]:
# import dataset
with ZipFile('spam_detection_dataset.zip') as z:
    with z.open(r'spam_email_dataset.csv') as f:
        df = pd.read_csv(f)
        
print('Read the data successfully')

Read the data successfully


In [4]:
df.head()

,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0


In [5]:
# shape of dataset
print(f'Shape of dataset: {df.shape}')

Shape of dataset: (10000, 20)


In [6]:
# checking of null values
def null_check(dataset):
    null_count = df['email_text'].isnull().sum()
    
    if 50 > null_count > 0 :
        print(f'There are {null_count} null values\n\nDeleting null values...\n\nShape of dataset before droping null values: {df.shape}')
        df.dropna(inplace=True)
        print(f'Shape of dataset before droping null values: {df.shape}')
    elif null_count > 50:
        print(f'There are {null_count} null values. Need some supervision')
    else:
        print('No null values')
    

null_check(df)

No null values


In [7]:
# distribution of target feature
print('Distribution count of target feature')
print(df['label'].value_counts())

print('\nDistribution percentage of target feature')
print(df['label'].value_counts()/df.shape[0] * 100)

Distribution count of target feature
label
0    6005
1    3995
Name: count, dtype: int64

Distribution percentage of target feature
label
0    60.05
1    39.95
Name: count, dtype: float64


In [8]:
# mapper
label2id = {'ham': 0, 'spam': 1}
id2label = {0: 'ham', 1: 'spam'}
r"\d+\.\d+|\w+|[^\w\s]"

'\\d+\\.\\d+|\\w+|[^\\w\\s]'

In [17]:
tokenizer = RegexpTokenizer(r"\d+\.\d+|\w+|[^\w\s]")
lammetizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        print('Not string type')
        return []

    # change to lowercase
    text = text.lower()
    
    # replace all urls with urltoken
    text = regex.sub(r'https?://\S+|www.S+', 'urltoken', text)
    # replace all the usernames mention with usertoken
    text = regex.sub(r'@\w+', 'usertoken', text)
    # replace all emoji with emojitoken
    text = emoji.replace_emoji(text, replace='emojitoken')
    
    # tokenize
    tokens = tokenizer.tokenize(text)
    
    # lemmaitize
    clean_tokens = [lammetizer.lemmatize(token) for token in tokens if len(token)>1 and not token.isdigit()]
    
    return clean_tokens

In [18]:
print(preprocess_text("Hello this is @PrashantKumar! I'm 😊. This is a good website https://sxioalsajie.sidoa. 👋🏼"))

['hello', 'this', 'is', 'usertoken', 'emojitoken', 'this', 'is', 'good', 'website', 'urltoken', 'emojitoken']
